#### Aspect Keywords from Literature

In [ ]:
staff_keywords = [

    "staff",
    "service",
    "employee",
    "employees",
    "personnel",
    "team",
    "worker",

    "reception",
    "receptionist",
    "front desk",
    "check in",
    "check-in",

    "manager",
    "host",
    "hostess",
    "concierge",

    "friendly",
    "helpful",
    "welcoming",
    "professional",
    "courteous",

    "customer service",
    "hospitality"
]

In [ ]:
cleanliness_keywords = [

    "clean",
    "cleanliness",
    "spotless",
    "tidy",

    "dirty",
    "messy",
    "dust",
    "dusty",

    "hygiene",
    "sanitary",

    "bathroom",
    "toilet",
    "shower",

    "bed",
    "bedsheets",
    "linen",
    "towel",

    "stain",
    "smell",
    "odour",
    "odor"
]

In [ ]:
location_keywords = [

    "location",
    "distance",

    "city centre",
    "city center",
    "downtown",
    "city",
    "center",
    "centre",

    "airport",
    "station",
    "bus",

    "walking distance",
    "nearby",

    "beach",
    "sea",
    "view",
    "lake",
    "pool",
    "ocean",
    "waterfall",

    "transport",
    "access",

    "central",
    "convenient",

    "parking",
    "park",
    "mall",
    "shopping",
]

In [ ]:
value_keywords = [

    "value",
    "value for money",

    "price",
    "pricing",
    "cost",

    "expensive",
    "cheap",

    "affordable",
    "budget",

    "worth",
    "worthwhile",

    "overpriced",
    "reasonable",

    "money",
    "paid",
    "rate"
]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

processed_path = "/content/drive/MyDrive/Colab Notebooks/Irish Hotel Reviews/Dataset/processed/irish_hotel_reviews_processed.csv"

df = pd.read_csv(processed_path)

print(df.shape)
print(df.columns)

(13294, 13)
Index(['review_id', 'review_text', 'overall_rating', 'hotel_name', 'county',
       'trip_type', 'language', 'clean_review', 'word_count', 'sentiment',
       'anonymized_review', 'final_review', 'clean_review_ml'],
      dtype='object')


#### Generate candidate keywords using TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(df["clean_review_ml"])

terms = vectorizer.get_feature_names_out()

tfidf_scores = X.sum(axis=0).A1

tfidf_df = pd.DataFrame({
    "term": terms,
    "score": tfidf_scores
})

tfidf_df = tfidf_df.sort_values(
    by="score",
    ascending=False
)

print(tfidf_df.head(100))

       term       score
416   hotel  825.506130
623  person  788.579449
719    room  680.098653
804   staff  665.199572
366   great  620.997009
..      ...         ...
825    star  114.029365
559  minute  113.818496
940    view  113.793169
204   didnt  113.158666
155    come  112.427519

[100 rows x 2 columns]


In [ ]:
tfidf_df.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Irish Hotel Reviews/Dataset/candidate_keywords_tfidf.csv",
    index=False
)

#### Build Word2Vec Model

In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.6 MB/s eta 0:00:00


In [ ]:
# train the model
from gensim.models import Word2Vec

sentences = [
    review.split()
    for review in df["clean_review_ml"]
]

model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4
)

In [ ]:
model.save("/content/drive/MyDrive/Colab Notebooks/Irish Hotel Reviews/Dataset/word2vec_model/hotel_reviews_word2vec.model")

In [ ]:
model.wv.most_similar(
    "clean",
    topn=20
)

[('spotless', 0.921722412109375),
 ('spacious', 0.8805510997772217),
 ('tidy', 0.8728525042533875),
 ('spotlessly', 0.8556082248687744),
 ('modern', 0.846021831035614),
 ('stylish', 0.8404029607772827),
 ('cozy', 0.8387376666069031),
 ('nicely', 0.8371930718421936),
 ('appointed', 0.8321655988693237),
 ('bright', 0.8193922638893127),
 ('cosy', 0.8128646016120911),
 ('immaculate', 0.8098204135894775),
 ('equipped', 0.8041330575942993),
 ('maintained', 0.7992038130760193),
 ('quiet', 0.7984271049499512),
 ('decorated', 0.7913798093795776),
 ('decor', 0.7851454019546509),
 ('designed', 0.7839025259017944),
 ('sized', 0.7675423622131348),
 ('large', 0.7536672949790955)]

#### Seed words

##### Staff

In [ ]:
staff_seed_words = [
    "staff",
    "service",
    "reception"
]

In [ ]:
cleanliness_seed_words = [
    "clean",
    "cleanliness",
    "dirty"
]

In [ ]:
location_seed_words = [
    "location",
    "airport",
    "centre"
]

In [ ]:
value_seed_words = [
    "value",
    "price",
    "cost"
]

In [ ]:
import pandas as pd

staff_results = []

for word in staff_seed_words:
    results = model.wv.most_similar(word, topn=20)

    for discovered_word, similarity in results:
        staff_results.append({
            "aspect":"staff",
            "seed_word": word,
            "discovered_word": discovered_word,
            "similarity": similarity
        })

staff_df = pd.DataFrame(staff_results)

print(staff_df)

   aspect  seed_word discovered_word  similarity
0   staff      staff       efficient    0.693065
1   staff      staff   knowledgeable    0.690846
2   staff      staff    professional    0.678560
3   staff      staff          polite    0.669352
4   staff      staff          chatty    0.664398
5   staff      staff       extremely    0.653992
6   staff      staff       christina    0.646219
7   staff      staff           super    0.645593
8   staff      staff   accommodating    0.643687
9   staff      staff       courteous    0.642545
10  staff      staff        employee    0.631708
11  staff      staff      incredibly    0.627253
12  staff      staff    approachable    0.623318
13  staff      staff   exceptionally    0.621951
14  staff      staff    particularly    0.619675
15  staff      staff        everyone    0.608271
16  staff      staff      personable    0.607538
17  staff      staff        freindly    0.605972
18  staff      staff            kind    0.605677
19  staff      staff

In [ ]:
import pandas as pd

cleanliness_results = []

for word in cleanliness_seed_words:
    results = model.wv.most_similar(word, topn=20)

    for discovered_word, similarity in results:
        cleanliness_results.append({
            "aspect":"cleanliness",
            "seed_word": word,
            "discovered_word": discovered_word,
            "similarity": similarity
        })

cleanliness_df = pd.DataFrame(cleanliness_results)

print(cleanliness_df)

         aspect    seed_word discovered_word  similarity
0   cleanliness        clean        spotless    0.921722
1   cleanliness        clean        spacious    0.880551
2   cleanliness        clean            tidy    0.872853
3   cleanliness        clean      spotlessly    0.855608
4   cleanliness        clean          modern    0.846022
5   cleanliness        clean         stylish    0.840403
6   cleanliness        clean            cozy    0.838738
7   cleanliness        clean          nicely    0.837193
8   cleanliness        clean       appointed    0.832166
9   cleanliness        clean          bright    0.819392
10  cleanliness        clean            cosy    0.812865
11  cleanliness        clean      immaculate    0.809820
12  cleanliness        clean        equipped    0.804133
13  cleanliness        clean      maintained    0.799204
14  cleanliness        clean           quiet    0.798427
15  cleanliness        clean       decorated    0.791380
16  cleanliness        clean   

In [ ]:
import pandas as pd

cleanliness_results = []

for word in cleanliness_seed_words:
    results = model.wv.most_similar(word, topn=20)

    for discovered_word, similarity in results:
        cleanliness_results.append({
            "aspect":"cleanliness",
            "seed_word": word,
            "discovered_word": discovered_word,
            "similarity": similarity
        })

cleanliness_df = pd.DataFrame(cleanliness_results)

print(cleanliness_df)

         aspect    seed_word discovered_word  similarity
0   cleanliness        clean        spotless    0.921722
1   cleanliness        clean        spacious    0.880551
2   cleanliness        clean            tidy    0.872853
3   cleanliness        clean      spotlessly    0.855608
4   cleanliness        clean          modern    0.846022
5   cleanliness        clean         stylish    0.840403
6   cleanliness        clean            cozy    0.838738
7   cleanliness        clean          nicely    0.837193
8   cleanliness        clean       appointed    0.832166
9   cleanliness        clean          bright    0.819392
10  cleanliness        clean            cosy    0.812865
11  cleanliness        clean      immaculate    0.809820
12  cleanliness        clean        equipped    0.804133
13  cleanliness        clean      maintained    0.799204
14  cleanliness        clean           quiet    0.798427
15  cleanliness        clean       decorated    0.791380
16  cleanliness        clean   

In [ ]:
import pandas as pd

location_results = []

for word in location_seed_words:
    results = model.wv.most_similar(word, topn=20)

    for discovered_word, similarity in results:
        location_results.append({
            "aspect":"location",
            "seed_word": word,
            "discovered_word": discovered_word,
            "similarity": similarity
        })

location_df = pd.DataFrame(location_results)

print(location_df)

      aspect seed_word discovered_word  similarity
0   location  location         located    0.742088
1   location  location            spot    0.726551
2   location  location       exploring    0.726237
3   location  location        situated    0.704982
4   location  location         central    0.701141
5   location  location            base    0.698682
6   location  location          centre    0.693589
7   location  location      convenient    0.692217
8   location  location          center    0.690344
9   location  location       centrally    0.674815
10  location  location           heart    0.673085
11  location  location    conveniently    0.669460
12  location  location           close    0.667245
13  location  location            town    0.664436
14  location  location          dublin    0.653359
15  location  location      attraction    0.645925
16  location  location            city    0.645026
17  location  location           arena    0.639297
18  location  location        s

In [ ]:
import pandas as pd

value_results = []

for word in value_seed_words:
    results = model.wv.most_similar(word, topn=20)

    for discovered_word, similarity in results:
        value_results.append({
            "aspect":"value",
            "seed_word": word,
            "discovered_word": discovered_word,
            "similarity": similarity
        })

value_df = pd.DataFrame(value_results)

print(value_df)

   aspect seed_word discovered_word  similarity
0   value     value           money    0.826807
1   value     value           price    0.777317
2   value     value      reasonable    0.712922
3   value     value         overall    0.698216
4   value     value            rate    0.669139
5   value     value          priced    0.649030
6   value     value        location    0.634024
7   value     value         quality    0.632414
8   value     value      reasonably    0.632087
9   value     value         amenity    0.629738
10  value     value           liked    0.608518
11  value     value     welllocated    0.602327
12  value     value   accommodation    0.601532
13  value     value           range    0.593455
14  value     value           worth    0.586040
15  value     value     considering    0.583582
16  value     value      affordable    0.579804
17  value     value       heartbeat    0.567982
18  value     value     cleanliness    0.566824
19  value     value        facility    0

In [ ]:
combined_df = pd.concat(
    [staff_df, cleanliness_df, location_df, value_df],
    axis=0,          # stack rows (default)
    ignore_index=True
)

print(combined_df.head())

  aspect seed_word discovered_word  similarity
0  staff     staff       efficient    0.693065
1  staff     staff   knowledgeable    0.690846
2  staff     staff    professional    0.678560
3  staff     staff          polite    0.669352
4  staff     staff          chatty    0.664398


In [ ]:
combined_df.shape

(240, 4)

In [ ]:
combined_df = (
    combined_df
    .sort_values("similarity", ascending=False)
    .drop_duplicates(subset="discovered_word")
    .reset_index(drop=True)
)

In [ ]:
combined_df.head()

,aspect,seed_word,discovered_word,similarity
0,cleanliness,dirty,replaced,0.964816
1,location,airport,shuttle,0.959588
2,cleanliness,dirty,tap,0.958352
3,cleanliness,dirty,properly,0.957378
4,cleanliness,dirty,fan,0.957189


In [ ]:
combined_df.to_csv(
    "aspect_expansion_table.csv",
    index=False
)

In [ ]:
staff_word2vec_keywords = [
    "receptionist",
    "gentleman",
    "doorman",
    "young",
    "man",
    "concierge",
    "checking",
    "lady",
    "dealt",
    "greeted",
    "chatty",
    "met",
    "kind",
    "greeting",
    "courteous",
    "checkin",
    "server",
    "polite",
    "efficient",
    "incredibly",
    "alberto",
    "extremely",
    "approachable",
    "waiter",
    "employee",
    "knowledgeable",
    "professional",
    "consistently",
    "attention",
    "presentation",
    "particular",
    "encountered",
    "job",
    "everyone",
    "exceptionally",
    "bartender",
    "hospitality",
    "smiley",
    "prompt",
    "cleanliness",
    "host",
    "accommodating",
    "top",
    "customer",
    "team",
    "personable"
]

In [ ]:
cleanliness_word2vec_keywords = [
    "fan",
    "hair",
    "toilet",
    "tap",
    "light",
    "heat",
    "wash",
    "broken",
    "carpet",
    "mirror",
    "sink",
    "clothes",
    "properly",
    "ac",
    "curtain",
    "spotless",
    "spacious",
    "tidy",
    "comfort",
    "spotlessly",
    "high",
    "quality",
    "cozy",
    "modern",
    "bright",
    "standard",
    "amenity",
    "cosy",
    "equipped",
    "immaculate",
    "decor",
    "maintained",
    "elegant",
    "comfy",
    "luxurious",
    "interior",
    "impressive",
    "accommodation",
    "importantly",
    "wellappointed",
    "aesthetically",
]

In [ ]:
location_word2vec_keywords = [
    "center",
    "shuttle",
    "bus",
    "stop",
    "dart",
    "hop",
    "exploring",
    "tram",
    "train",
    "handy",
    "right",
    "positioned",
    "ride",
    "transport",
    "across",
    "downtown",
    "explore",
    "transportation",
    "sight",
    "road",
    "arena",
    "close",
    "walkable",
    "reach",
    "near",
    "situated",
    "town",
    "central",
    "heart",
    "attraction",
    "stroll",
    "stadium",
    "shopping",
    "convenient",
    "everywhere",
    "located",
    "base",
    "spot",
    "centre",
    "centrally",
    "city",
    "walkability",
]

In [ ]:
value_word2vec_keywords = [
    "avoid",
    "paying",
    "laundry",
    "tired",
    "slightly",
    "least",
    "worried",
    "vent",
    "reasonable",
    "expensive",
    "money",
    "rate",
    "considering",
    "expect",
    "offer",
    "price",
    "value",
    "expected",
    "fine",
    "decent",
    "priced",
    "overall",
    "outrageous",
    "compared",
    "average",
    "pricey",
    "reasonably",
    "choice",
    "range",
    "picky",
    "facility",
    "welllocated",
    "variety",
    "size"
]

In [ ]:
staff_combined = (
    pd.Series(staff_keywords + staff_word2vec_keywords)
    .drop_duplicates()
    .tolist()
)

staff_combined

['staff',
 'service',
 'employee',
 'employees',
 'personnel',
 'team',
 'worker',
 'reception',
 'receptionist',
 'front desk',
 'check in',
 'check-in',
 'manager',
 'host',
 'hostess',
 'concierge',
 'friendly',
 'helpful',
 'welcoming',
 'professional',
 'courteous',
 'customer service',
 'hospitality',
 'gentleman',
 'doorman',
 'young',
 'man',
 'checking',
 'lady',
 'dealt',
 'greeted',
 'chatty',
 'met',
 'kind',
 'greeting',
 'checkin',
 'server',
 'polite',
 'efficient',
 'incredibly',
 'alberto',
 'extremely',
 'approachable',
 'waiter',
 'knowledgeable',
 'consistently',
 'attention',
 'presentation',
 'particular',
 'encountered',
 'job',
 'everyone',
 'exceptionally',
 'bartender',
 'smiley',
 'prompt',
 'cleanliness',
 'accommodating',
 'top',
 'customer',
 'personable']

In [ ]:
import json

# Map aspect name -> list pair
aspect_sources = {
    "staff": [staff_keywords, staff_word2vec_keywords],
    "cleanliness": [cleanliness_keywords, cleanliness_word2vec_keywords],
    "location": [location_keywords, location_word2vec_keywords],
    "value": [value_keywords, value_word2vec_keywords],
}

# Combine and deduplicate
aspect_dictionary = {
    aspect: (
        pd.Series(lst1 + lst2)
        .drop_duplicates()
        .tolist()
    )
    for aspect, (lst1, lst2) in aspect_sources.items()
}

# Save to JSON
with open("aspect_dictionary.json", "w", encoding="utf-8") as f:
    json.dump(aspect_dictionary, f, indent=4, ensure_ascii=False)

print("Saved to aspect_dictionary.json")

Saved to aspect_dictionary.json


#### Create Aspect Dictionary Pipeline

In [ ]:
review = "Staff were extremely helpful. The room was spotless. Location was a little far from town."

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(review)

In [ ]:
sentences

['Staff were extremely helpful.',
 'The room was spotless.',
 'Location was a little far from town.']

In [ ]:
import json

with open("aspect_dictionary.json") as f:
    aspect_dict = json.load(f)

In [ ]:
def detect_aspects(sentence):

    sentence_lower = sentence.lower()

    found_aspects = []

    for aspect, keywords in aspect_dict.items():

        for keyword in keywords:

            if keyword in sentence_lower:

                found_aspects.append(aspect)

                break

    return found_aspects

In [ ]:
sentence = "Staff were extremely friendly"

print(detect_aspects(sentence))

['staff']


In [ ]:
review = """
Our stay in the hotel was a real pleasure in this hotel.
Thanks to Elena in reception that suggested us a very good places to visit the town.
"""

In [ ]:
records = []

sentences = sent_tokenize(review)

for sentence in sentences:

    aspects = detect_aspects(sentence)

    for aspect in aspects:

        records.append({
            "aspect": aspect,
            "text": sentence
        })

In [ ]:
records

[{'aspect': 'staff',
  'text': 'Thanks to Elena in reception that suggested us a very good places to visit the town.'},
 {'aspect': 'cleanliness',
  'text': 'Thanks to Elena in reception that suggested us a very good places to visit the town.'},
 {'aspect': 'location',
  'text': 'Thanks to Elena in reception that suggested us a very good places to visit the town.'}]

#### process entire dataset

In [ ]:
aspect_records = []

for idx, row in df.iterrows():

    review_id = row["review_id"]

    review = row["final_review"]

    sentences = sent_tokenize(review)

    for sentence in sentences:

        aspects = detect_aspects(sentence)

        for aspect in aspects:

            aspect_records.append({

                "review_id": review_id,

                "aspect": aspect,

                "aspect_text": sentence
            })

In [ ]:
aspect_df = pd.DataFrame(
    aspect_records
)

In [ ]:
aspect_df.head()

,review_id,aspect,aspect_text
0,1062725118,location,This hotel was a great location.
1,1062725118,cleanliness,"The rooms were spacious for Europe, and it had..."
2,1062725118,cleanliness,"The vending machine was out of order, there is..."
3,1062725118,staff,So plan to stop at the local food shops for wa...
4,1062725118,location,So plan to stop at the local food shops for wa...


In [ ]:
aspect_df.shape

(69520, 3)

In [ ]:
aspect_df.to_csv(
    "rule_based_aspects.csv",
    index=False
)

#### Create a dataset for manual annotation

In [ ]:
sample_df = df.sample(
    500,
    random_state=42
)

In [ ]:
sample_df.to_csv(
    "manual_annotation.csv",
    index=False
)